# 07 – Statistical Validation

In this notebook, I check whether recharge behaviour and tenure differ between churned and retained subscribers. I also compare churn rates across recharge-frequency groups and perform a robustness check.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

con = sqlite3.connect(r'..\\data\\raw\\airtel.db')

def q(sql):
    return pd.read_sql(sql, con)


## Subscriber data

I use `customer_features` and recharge transactions to create one row per subscriber. Subscribers with no recharge activity are kept.

In [2]:
q("""
SELECT COUNT(*) AS subscribers,
       SUM(is_churned) AS churned
FROM customer_features
""")

,subscribers,churned
0,22000,5226


In [3]:
recharge = q("""
SELECT customer_id,
       SUM(amount_inr) AS total_recharge,
       COUNT(*) AS recharge_count
FROM recharges
WHERE recharge_date >= '2025-01-01'
  AND recharge_date < '2025-07-01'
GROUP BY customer_id
""")

# Start with the subscriber table
validation_df = q("SELECT * FROM customer_features")

# Remove recharge columns already in customer_features,
# so the merge does not create recharge_count_x / recharge_count_y
validation_df = validation_df.drop(
    columns=["recharge_count", "recharge_total_6m", "actual_arpu_6m"],
    errors="ignore"
)

# Add recharge totals and counts for the analysis period
validation_df = validation_df.merge(recharge, on="customer_id", how="left")

validation_df["total_recharge"] = validation_df["total_recharge"].fillna(0)
validation_df["recharge_count"] = validation_df["recharge_count"].fillna(0)

# Average recharge per transaction
validation_df["average_recharge"] = (
    validation_df["total_recharge"] / validation_df["recharge_count"]
)
validation_df["average_recharge"] = validation_df["average_recharge"].replace(
    [np.inf, -np.inf], np.nan
)

validation_df.head()

,customer_id,activation_date,tenure_days,tenure_years,observed_days,circle_name,region,plan_name,plan_type,price_inr,...,recency_days,stated_arpu,arpu_gap,complaint_count,unresolved_count,resolved_count,avg_resolution_days,total_recharge,recharge_count,average_recharge
0,9,2022-09-04,1030,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,2.0,374,83.17,3,0,3,9.33,1745.0,5.0,349.0
1,31,2024-02-10,506,1,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,1.0,257,17.33,0,0,0,NaN,1438.0,2.0,719.0
2,151,2022-07-22,1074,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,5.0,374,25.00,4,1,3,5.33,2094.0,6.0,349.0
3,186,2022-01-04,1273,3,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,0.0,257,17.33,0,0,0,NaN,1438.0,2.0,719.0
4,207,2024-12-14,198,0,91,Delhi,North,Prepaid Value 209,Prepaid,209,...,89.0,224,154.33,3,0,3,5.33,418.0,2.0,209.0


## Task 1 – Churned vs retained

I compare the average values of churned and retained subscribers. Welch's t-test is used to test the difference between the means.

In [4]:
churned = validation_df[validation_df["is_churned"] == 1]
retained = validation_df[validation_df["is_churned"] == 0]

print("Total recharge")
print("Churned mean:", churned["total_recharge"].mean())
print("Retained mean:", retained["total_recharge"].mean())
stats.ttest_ind(churned["total_recharge"], retained["total_recharge"], equal_var=False)


Total recharge
Churned mean: 928.0757749712974
Retained mean: 2421.5538333134614


TtestResult(statistic=np.float64(-53.21411407819679), pvalue=np.float64(0.0), df=np.float64(16921.492658376148))

In [5]:
print("Recharge count")
print("Churned mean:", churned["recharge_count"].mean())
print("Retained mean:", retained["recharge_count"].mean())
stats.ttest_ind(churned["recharge_count"], retained["recharge_count"], equal_var=False)


Recharge count
Churned mean: 2.015499425947187
Retained mean: 4.361511863598426


TtestResult(statistic=np.float64(-64.1802273779107), pvalue=np.float64(0.0), df=np.float64(11289.580891844178))

In [6]:
churned_active = churned[churned["recharge_count"] > 0]
retained_active = retained[retained["recharge_count"] > 0]

print("Average recharge per transaction (recharge-active subscribers)")
print("Churned mean:", churned_active["average_recharge"].mean())
print("Retained mean:", retained_active["average_recharge"].mean())

stats.ttest_ind(churned_active["average_recharge"],
                retained_active["average_recharge"],
                equal_var=False)


Average recharge per transaction (recharge-active subscribers)
Churned mean: 515.4687465103294
Retained mean: 593.6211044627274


TtestResult(statistic=np.float64(-9.821620177513193), pvalue=np.float64(1.3919805112852763e-22), df=np.float64(5503.720788617845))

In [7]:
print("Tenure")
print("Churned mean:", churned["tenure_days"].mean())
print("Retained mean:", retained["tenure_days"].mean())
stats.ttest_ind(churned["tenure_days"], retained["tenure_days"], equal_var=False)


Tenure
Churned mean: 651.5876387294297
Retained mean: 711.8798736139263


TtestResult(statistic=np.float64(-11.54884292689635), pvalue=np.float64(1.244091060345417e-30), df=np.float64(8788.463708617102))

## Task 2 – Recharge frequency groups

I divide subscribers into Low (0–2), Medium (3–5) and High (6 or more) recharge groups and calculate the churn rate.

In [8]:
validation_df["frequency_group"] = "High"
validation_df.loc[validation_df["recharge_count"] <= 2, "frequency_group"] = "Low"
validation_df.loc[(validation_df["recharge_count"] >= 3) &
                  (validation_df["recharge_count"] <= 5), "frequency_group"] = "Medium"

frequency_summary = validation_df.groupby("frequency_group")["is_churned"].agg(
    subscribers="count", churned="sum"
).reset_index()

frequency_summary["churn_rate"] = (
    frequency_summary["churned"] / frequency_summary["subscribers"]
)

frequency_summary

,frequency_group,subscribers,churned,churn_rate
0,High,8041,246,0.030593
1,Low,8812,3413,0.387313
2,Medium,5147,1567,0.304449


### Low vs High group test

A two-proportion Z-test compares the churn rates in the Low and High groups.

In [9]:
low = validation_df[validation_df["frequency_group"] == "Low"]
high = validation_df[validation_df["frequency_group"] == "High"]

counts = [low["is_churned"].sum(), high["is_churned"].sum()]
sizes = [len(low), len(high)]

proportions_ztest(counts, sizes)


(np.float64(56.10329223344183), np.float64(0.0))

## Task 3 – Tenure

The tenure test is included in Task 1 above. It compares tenure days for churned and retained subscribers using Welch's t-test.

## Task 4 – Robustness check

I repeat the total-recharge comparison after removing subscribers who had no recharge activity.

In [10]:
churned_active = validation_df[(validation_df["is_churned"] == 1) &
                               (validation_df["recharge_count"] > 0)]
retained_active = validation_df[(validation_df["is_churned"] == 0) &
                                (validation_df["recharge_count"] > 0)]

print("Active churned mean:", churned_active["total_recharge"].mean())
print("Active retained mean:", retained_active["total_recharge"].mean())

stats.ttest_ind(churned_active["total_recharge"],
                retained_active["total_recharge"],
                equal_var=False)


Active churned mean: 1354.0268006700167
Active retained mean: 2761.8918882164958


TtestResult(statistic=np.float64(-42.32270453235038), pvalue=np.float64(0.0), df=np.float64(9739.78638918568))

## Export files

I save the dataset and frequency summary to the project's `reports` folder.

In [11]:
from pathlib import Path

out_dir = Path("../reports")
out_dir.mkdir(parents=True, exist_ok=True)

validation_df.to_csv(out_dir / "validation_df.csv", index=False)
frequency_summary.to_excel(out_dir / "07_Frequency_Summary.xlsx", index=False)

print("Files saved in:", out_dir.resolve())

Files saved in: C:\Users\navee\OneDrive\Desktop\airtel-telecom-analytics\reports


## Conclusion

Write a short conclusion based on the test results. Mention the group differences, frequency comparison and robustness check. Statistical association does not prove causation.